# nb_ingest_bronze
Generic, metadata-driven ingestion for CSV and JSON sources. Pipeline passes `entity` (and optionally `batch_id`).
Attach formula1 as the default lakehouse.

In [5]:
# PARAMETERS CELL (toggle "parameter cell" on this cell in Fabric so the pipeline can override)
entity = "circuits"
batch_id = ""

StatementMeta(, 71c20189-792d-4c57-9000-e3505aa4f171, 7, Finished, Available, Finished, False)

In [6]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

batch_id = batch_id or str(uuid.uuid4())
start_ts = datetime.utcnow()

# ---------- 1. read config ----------
cfg = (spark.table("meta_entities")
            .filter((F.col("entity") == entity) & (F.col("is_active") == 1))
            .first())
if cfg is None:
    raise ValueError(f"Entity '{entity}' not found or inactive in meta_entities")

status, rows_written, error_msg = "SUCCESS", None, None

try:
    # ---------- 2. read source (csv: all strings; json: Spark-inferred, nested structs preserved) ----------
    fmt = cfg.file_format.lower()
    reader = spark.read.format(fmt)

    if fmt == "csv":
        reader = (reader.option("header", str(bool(cfg.header)).lower())
                        .option("inferSchema", "false"))
        if cfg.delimiter:  reader = reader.option("delimiter", cfg.delimiter)
        if cfg.null_value: reader = reader.option("nullValue", cfg.null_value)
    elif fmt == "json":
        if cfg.multiline:  reader = reader.option("multiLine", "true")
    else:
        raise ValueError(f"Unsupported file_format '{fmt}'")

    df = reader.load(cfg.source_path)      # works for a single file or a folder

    # ---------- 3. audit columns (same names as the original Databricks project) ----------
    df = (df.withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("source_file",         F.col("_metadata.file_path"))
            .withColumn("batch_id",            F.lit(batch_id)))

    # ---------- 4. write bronze delta ----------
    (df.write.format("delta")
       .mode(cfg.write_mode)
       .option("overwriteSchema", "true" if cfg.write_mode == "overwrite" else "false")
       .saveAsTable(cfg.bronze_table))

    rows_written = spark.table(cfg.bronze_table).filter(F.col("batch_id") == batch_id).count()
    print(f"[{entity}] wrote {rows_written} rows to {cfg.bronze_table}")

except Exception as e:
    status, error_msg = "FAILED", str(e)[:4000]
    raise

finally:
    # ---------- 5. log ----------
    log = spark.createDataFrame(
        [(batch_id, entity, "bronze", status, rows_written, start_ts, datetime.utcnow(), error_msg)],
        "batch_id string, entity string, layer string, status string, rows_written long, "
        "start_ts timestamp, end_ts timestamp, error_msg string")
    log.write.format("delta").mode("append").saveAsTable("meta_run_log")

StatementMeta(, 71c20189-792d-4c57-9000-e3505aa4f171, 8, Finished, Available, Finished, False)

/tmp/ipykernel_6410/1778090962.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow()
/tmp/ipykernel_6410/1778090962.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(batch_id, entity, "bronze", status, rows_written, start_ts, datetime.utcnow(), error_msg)],


In [7]:
display(spark.table(cfg.bronze_table))

StatementMeta(, 71c20189-792d-4c57-9000-e3505aa4f171, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9cb6b9ab-c68d-4417-9865-4f9976541259)